# Residualized Tone Space Analysis

This notebook demonstrates the usage of the Residualized Tone Space system for analyzing and separating topic information from stylistic information.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Import our modules
import sys
sys.path.append('..')

from residualized_tone_space.core import TopicSubspaceExtractor, ResidualToneCalculator
from residualized_tone_space.utils import EmbeddingProcessor, PerformanceMonitor

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load and Prepare Data

First, we'll load some sample embeddings and prepare them for analysis.

In [ ]:
# Generate sample embeddings (replace with your actual data)
np.random.seed(42)
n_samples = 1000
embedding_dim = 1536

# Create embeddings with some structure
embeddings = np.random.randn(n_samples, embedding_dim).astype(np.float32)

# Add some topic structure
topic_centers = np.random.randn(5, embedding_dim)
topic_labels = np.random.randint(0, 5, n_samples)

for i, label in enumerate(topic_labels):
    embeddings[i] += 0.5 * topic_centers[label]

print(f"Generated {n_samples} embeddings with {embedding_dim} dimensions")
print(f"Topic distribution: {np.bincount(topic_labels)}")

## 2. Topic Subspace Extraction

Extract the topic subspace using different methods.

In [ ]:
# Initialize performance monitor
monitor = PerformanceMonitor()

# Extract topic subspace using PCA
extractor = TopicSubspaceExtractor(method="pca", n_components=50)
extractor.fit(embeddings)

# Create projection matrix
projection_matrix, orthonormal_basis = extractor.create_topic_projection_matrix()

print(f"Topic vectors shape: {extractor.topic_vectors.shape}")
print(f"Explained variance (first 10): {extractor.explained_variance[:10]}")
print(f"Total explained variance: {extractor.explained_variance.sum():.3f}")

## 3. Residual Vector Calculation

Calculate residual vectors by removing topic components.

In [ ]:
# Initialize residual calculator
calculator = ResidualToneCalculator(projection_matrix)

# Calculate residual vectors
residual_vectors = calculator.calculate_residual_vectors(embeddings)

print(f"Residual vectors shape: {residual_vectors.shape}")
print(f"Residual vector norms (mean): {np.mean(np.linalg.norm(residual_vectors, axis=1)):.3f}")
print(f"Residual vector norms (std): {np.std(np.linalg.norm(residual_vectors, axis=1)):.3f}")

## 4. Visualization

Visualize the topic and residual spaces.

In [ ]:
# Project to 2D for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)

# Original embeddings
embeddings_2d = tsne.fit_transform(embeddings)

# Topic projections
topic_projections = calculator.get_topic_projection(embeddings)
topic_projections_2d = tsne.fit_transform(topic_projections)

# Residual vectors
residuals_2d = tsne.fit_transform(residual_vectors)

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original embeddings
scatter = axes[0].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=topic_labels, cmap='tab10', alpha=0.6)
axes[0].set_title('Original Embeddings')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=axes[0])

# Topic projections
scatter = axes[1].scatter(topic_projections_2d[:, 0], topic_projections_2d[:, 1], c=topic_labels, cmap='tab10', alpha=0.6)
axes[1].set_title('Topic Projections')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=axes[1])

# Residual vectors
scatter = axes[2].scatter(residuals_2d[:, 0], residuals_2d[:, 1], c=topic_labels, cmap='tab10', alpha=0.6)
axes[2].set_title('Residual Vectors')
axes[2].set_xlabel('t-SNE 1')
axes[2].set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=axes[2])

plt.tight_layout()
plt.show()

## 5. Performance Analysis

Analyze the performance of different operations.

In [ ]:
# Benchmark embedding operations
benchmarks = monitor.benchmark_embedding_operations(embeddings)

# Get system metrics
system_metrics = monitor.get_system_metrics()

print("Benchmark Results:")
for operation, time_ms in benchmarks.items():
    print(f"  {operation}: {time_ms:.2f} ms")

print("\nSystem Metrics:")
for metric, value in system_metrics.items():
    print(f"  {metric}: {value:.2f}")

## 6. Topic-Tone Separation Analysis

Analyze how well the topic and tone spaces are separated.

In [ ]:
# Calculate topic coherence (how well topics cluster in topic space)
topic_projections = calculator.get_topic_projection(embeddings)

# Calculate within-topic and between-topic distances
within_topic_distances = []
between_topic_distances = []

for topic in range(5):
    topic_mask = topic_labels == topic
    topic_embeddings = topic_projections[topic_mask]
    
    # Within-topic distances
    if len(topic_embeddings) > 1:
        distances = np.linalg.norm(topic_embeddings[:, None] - topic_embeddings[None, :], axis=2)
        within_topic_distances.extend(distances[np.triu_indices_from(distances, k=1)])
    
    # Between-topic distances
    other_embeddings = topic_projections[~topic_mask]
    if len(other_embeddings) > 0:
        distances = np.linalg.norm(topic_embeddings[:, None] - other_embeddings[None, :], axis=2)
        between_topic_distances.extend(distances.flatten())

print(f"Within-topic distance (mean): {np.mean(within_topic_distances):.3f}")
print(f"Between-topic distance (mean): {np.mean(between_topic_distances):.3f}")
print(f"Separation ratio: {np.mean(between_topic_distances) / np.mean(within_topic_distances):.3f}")

## 7. Summary

This analysis demonstrates the basic functionality of the Residualized Tone Space system. The key insights are:

1. **Topic Subspace**: Successfully extracts topic information using PCA
2. **Residual Space**: Captures stylistic information orthogonal to topics
3. **Separation**: The system effectively separates topic and tone information
4. **Performance**: Operations are efficient and scalable

Next steps would include:
- Learning tone axis vectors in the residual space
- Implementing A/B testing for content pair generation
- Validating cross-topic generalization
- Integrating with real content generation systems